In [92]:
# Install Dependencies
%pip install langchain langchain-openai langchain-community openai
%pip install duckduckgo-search
%pip install tiktoken
%pip install Pillow
%pip install python-dotenv
%pip install pandas

3744.10s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


3750.06s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


3755.97s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


...

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


3767.82s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


3773.73s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [93]:
import os
import csv
import json
import pandas as pd
from datetime import datetime
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import Tool, tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage,
    trim_messages
)
from langchain_community.tools import DuckDuckGoSearchRun

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [94]:
# Initialize Language Model

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=2048
)

print("✅ LLM initialized: gpt-4o-mini")

✅ LLM initialized: gpt-4o-mini


In [95]:
# Search Tool

search = DuckDuckGoSearchRun()

@tool
def hospital_search_tool(query: str) -> str:
    """Search for nearby hospitals and medical facilities for a query."""
    try:
        enhanced_query = f"hospitals clinics medical facilities {query}"
        results = search.run(enhanced_query)
        return f"🏥 Hospital Search Results:\n{results}"
    except Exception as e:
        return f"❌ Search failed: {str(e)}"

print(hospital_search_tool.name)
print(hospital_search_tool.description)
print("\n🔍 Test Search:")
print(hospital_search_tool.invoke("hospitals near Kom Hamada, Behira, Egypt"))

hospital_search_tool
Search for nearby hospitals and medical facilities for a query.

🔍 Test Search:
🏥 Hospital Search Results:
January 19, 2026 - Healthcare in Egypt is based on a pluralistic system, comprising a variety of healthcare providers from the public as well as the private sector. The government ensures basic universal health coverage, although private services are also available for those with the ability to pay. March 1, 2026 - Following is a list of the top hospitals in Egypt as per information available with us. If you have a feedback to share, please use the comment section below 2 weeks ago - Global Medical City Hospital|** Global Medical City .. We care for your well being Med Right For Medical Services ** Our duties are to deliver patient-centricity, latest technology, and top-notch doctors.|MedPharo - Egypt health tourism portal May 5, 2025 - Download the list of Hospitals in Egypt. Smartscrapers provides an accurate directory and the latest data on the number of Ho

In [96]:
# CSV Storage Tool

CSV_FILE_PATH = "patient_cases.csv"

CSV_HEADERS = [
    "case_id",
    "timestamp",
    "patient_symptoms",
    "mri_description",
    "ai_summary",
    "recommended_actions",
    "disclaimer"
]

def initialize_csv():
    """Create CSV file with headers if it doesn't exist."""
    if not os.path.exists(CSV_FILE_PATH):
        with open(CSV_FILE_PATH, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=CSV_HEADERS)
            writer.writeheader()
        print(f"✅ CSV file created: {CSV_FILE_PATH}")
    else:
        print(f"📄 CSV file already exists: {CSV_FILE_PATH}")

initialize_csv()

@tool
def csv_storage_tool(case_data: str) -> str:
    """ Stores structured patient case data into a CSV file. Use this tool after generating a case summary to save the record. """
    try:
        data = json.loads(case_data)
        case_id = f"CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}"
        
        row = {
            "case_id": case_id,
            "timestamp": datetime.now().isoformat(),
            "patient_symptoms": data.get("patient_symptoms", "N/A"),
            "mri_description": data.get("mri_description", "N/A"),
            "ai_summary": data.get("ai_summary", "N/A"),
            "recommended_actions": data.get("recommended_actions", "N/A"),
            "disclaimer": "This is not a medical diagnosis. Consult a doctor."
        }
        
        with open(CSV_FILE_PATH, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=CSV_HEADERS)
            writer.writerow(row)
        
        return f"✅ Case saved successfully!\n📋 Case ID: {case_id}\n📁 File: {CSV_FILE_PATH}"
    
    except json.JSONDecodeError:
        return "❌ Error: Invalid JSON format. Please provide valid JSON."
    except Exception as e:
        return f"❌ Error saving case: {str(e)}"


test_data = json.dumps({
    "patient_symptoms": "Severe headache, blurred vision",
    "mri_description": "Mild white matter hyperintensities",
    "ai_summary": "Patient presents with neurological symptoms",
    "recommended_actions": "Consult neurologist"
})

print(csv_storage_tool.invoke(test_data))

📄 CSV file already exists: patient_cases.csv
✅ Case saved successfully!
📋 Case ID: CASE-20260424154214
📁 File: patient_cases.csv


In [97]:
# Verify stored data
df = pd.read_csv(CSV_FILE_PATH)
print("📊 Stored Patient Cases:")
print("=" * 60)
df

📊 Stored Patient Cases:


,case_id,timestamp,patient_symptoms,mri_description,ai_summary,recommended_actions,disclaimer
0,CASE-20260424145722,2026-04-24T14:57:22.590459,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
1,CASE-20260424145731,2026-04-24T14:57:31.308154,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
2,CASE-20260424145959,2026-04-24T14:59:59.546426,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
3,CASE-20260424150637,2026-04-24T15:06:37.689787,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
4,CASE-20260424151525,2026-04-24T15:15:25.900821,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
5,CASE-20260424152157,2026-04-24T15:21:57.890933,"Sharp chest pain during physical activity, Sho...",Left ventricular ejection fraction: 48% (mildl...,The patient presents with symptoms indicative ...,It is advisable for the patient to consult a c...,This is not a medical diagnosis. Consult a doc...
6,CASE-20260424152214,2026-04-24T15:22:14.428974,Persistent headache for 5 days (left temporal ...,Small area of white matter hyperintensity in t...,The patient presents with a combination of neu...,Consult a neurologist for further evaluation a...,This is not a medical diagnosis. Consult a doc...
7,CASE-20260424152321,2026-04-24T15:23:21.400827,"Persistent lower back pain for 2 weeks, pain r...",No MRI/Scan findings provided,The symptoms described suggest possible issues...,"Consult a healthcare professional, preferably ...",This is not a medical diagnosis. Consult a doc...
8,CASE-20260424152331,2026-04-24T15:23:31.332173,"Persistent lower back pain for 2 weeks, Pain r...",Not provided,The symptoms suggest possible nerve involvemen...,"Consult a healthcare professional, possibly a ...",This is not a medical diagnosis. Consult a doc...
9,CASE-20260424152635,2026-04-24T15:26:35.311464,Persistent headache for 5 days (left temporal ...,Small area of white matter hyperintensity in t...,"The patient's symptoms of persistent headache,...",Consult a neurologist for further evaluation a...,This is not a medical diagnosis. Consult a doc...


In [98]:
# Message Trimming for Context Optimization

def trim_conversation_messages(messages, max_tokens=2000):
    """ Trims conversation messages to stay within token limits. Keeps the system message and most recent messages. """

    trimmed = trim_messages(
        messages,
        max_tokens=max_tokens,
        strategy="last",
        token_counter=llm,
        include_system=True,
        start_on="human",
    )
    return trimmed

test_messages = [
    SystemMessage(content="You are a medical assistant. Provide helpful and accurate information based on patient symptoms and medical data."),
    HumanMessage(content="I have a headache. It's been getting worse over the past few days."),
    AIMessage(content="Can you describe the headache?, Please provide more details."),
    HumanMessage(content="It's throbbing and on the left side. Provide more details about the pain and any other symptoms you may have."),
    AIMessage(content="How long have you had it?"),
    HumanMessage(content="About 3 days now. Also, I've been feeling nauseous and have had some dizziness."),
]

trimmed = trim_conversation_messages(test_messages, max_tokens=100)
print(f"Original messages: {len(test_messages)}")
print(f"Trimmed messages:  {len(trimmed)}")
for msg in trimmed:
    print(f"  [{msg.__class__.__name__}]: {msg.content[:60]}...")

Original messages: 6
Trimmed messages:  4
  [SystemMessage]: You are a medical assistant. Provide helpful and accurate in...
  [HumanMessage]: It's throbbing and on the left side. Provide more details ab...
  [AIMessage]: How long have you had it?...
  [HumanMessage]: About 3 days now. Also, I've been feeling nauseous and have ...


In [99]:
# Summarization Buffer Memory

class SummaryMemory:
    def __init__(self, max_token_limit: int, memory_key: str, return_messages: bool = True, output_key: str = "output", llm=None):
        self.max_token_limit = max_token_limit
        self.memory_key = memory_key
        self.return_messages = return_messages
        self.output_key = output_key
        self.chat_history = []
        self.summary = ""
        self.llm = llm

    def load_memory_variables(self, inputs=None):
        if self.return_messages:
            return {self.memory_key: self.chat_history}
        return {self.memory_key: self.summary}

    def save_context(self, inputs, outputs):
        self.chat_history.append({"inputs": inputs, "outputs": outputs})

    def clear(self):
        self.chat_history.clear()
        self.summary = ""

memory = SummaryMemory(
    llm=llm,
    max_token_limit=500,
    return_messages=True,
    memory_key="chat_history",
    output_key="output"
 )

print("✅ Summarization Memory initialized")
print(f"   Max token limit: {memory.max_token_limit}")
print(f"   Memory key: {memory.memory_key}")

✅ Summarization Memory initialized
   Max token limit: 500
   Memory key: chat_history


In [100]:
# System Prompt & Agent Prompt Template

SYSTEM_PROMPT = """
You are a Multi-Modal Medical AI Assistant Agent. Your role is to help
analyze patient cases based on text symptoms and MRI/scan descriptions.

## YOUR CAPABILITIES:
1. **Analyze Inputs**: Process text symptoms and MRI/scan descriptions
2. **Generate Case Summary**: Create structured medical case summaries
3. **Provide Safe Medical Insights**: Offer general health information
4. **Hospital Search**: Find nearby hospitals when requested
5. **Store Case Data**: Save structured case records to CSV

## TOOL USAGE RULES:
- Use `hospital_search_tool` when the user asks about finding hospitals,
  clinics, or medical facilities
- Use `csv_storage_tool` after generating a complete case summary to
  store the structured data. Format the data as JSON with keys:
  patient_symptoms, mri_description, ai_summary, recommended_actions

## RESPONSE FORMAT:
When analyzing a case, structure your response as:

### 📋 Case Summary
- **Symptoms**: [list symptoms]
- **MRI/Scan Findings**: [describe findings]

### 🔍 Analysis
[Provide general medical insights about the symptoms and findings]

### 📌 Recommended Actions
[Suggest general next steps like consulting specialists]

### ⚠️ Disclaimer
*This is not a medical diagnosis. Consult a doctor.*

## STRICT CONSTRAINTS:
- ❌ NEVER provide a specific diagnosis
- ❌ NEVER prescribe medications
- ❌ NEVER replace professional medical advice
- ✅ ALWAYS include the disclaimer
- ✅ ALWAYS recommend consulting a healthcare professional
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ Prompt template created")

✅ Prompt template created


In [101]:
# Create Agent and Executor

# List of tools
tools = [hospital_search_tool, csv_storage_tool]

# Create the agent graph
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
 )

# Keep a backward-compatible name for the rest of the notebook
agent_executor = agent

print("✅ Agent created successfully!")
print(f"   Tools: {[t.name for t in tools]}")
print("   Memory: SummaryMemory")
print("   Max iterations: handled by create_agent")

✅ Agent created successfully!
   Tools: ['hospital_search_tool', 'csv_storage_tool']
   Memory: SummaryMemory
   Max iterations: handled by create_agent


In [102]:
# Multi-Modal Input Processing

class MedicalInputProcessor:
    def __init__(self):
        self.current_case = {
            "text_symptoms": None,
            "mri_description": None
        }
    
    def process_text_input(self, symptoms: str) -> str:
        self.current_case["text_symptoms"] = symptoms
        return symptoms
    
    def process_mri_description(self, mri_desc: str) -> str:
        self.current_case["mri_description"] = mri_desc
        return mri_desc
    
    def build_agent_input(
        self,
        text_symptoms: str,
        mri_description: Optional[str] = None,
        additional_request: Optional[str] = None
    ) -> str:
        self.process_text_input(text_symptoms)
        input_parts = []
        input_parts.append(f"## Patient Symptoms:\n{text_symptoms}")
        
        if mri_description:
            self.process_mri_description(mri_description)
            input_parts.append(f"\n## MRI/Scan Description:\n{mri_description}")
        
        if additional_request:
            input_parts.append(f"\n## Additional Request:\n{additional_request}")
        
        input_parts.append(
            "\nPlease analyze this case, provide a structured summary, "
            "and save the case data."
        )
        
        return "\n".join(input_parts)
    
    def reset(self):
        """Reset current case data."""
        self.current_case = {
            "text_symptoms": None,
            "mri_description": None
        }


input_processor = MedicalInputProcessor()
print("✅ MedicalInputProcessor initialized")

✅ MedicalInputProcessor initialized


In [103]:
# Test Case 1: Basic Symptom Analysis

print("=" * 100)
print("🧪 TEST CASE 1: Basic Symptom Analysis")
print("=" * 100)

# Build input
user_input = input_processor.build_agent_input(
    text_symptoms="""
    - Persistent headache for 5 days (left temporal region)
    - Blurred vision in left eye
    - Mild nausea in the morning
    - Difficulty concentrating
    - Patient age: 45, Male
    """,
    mri_description="""
    Brain MRI Report:
    - Small area of white matter hyperintensity in the left temporal lobe
    - No mass lesion identified
    - No midline shift
    - Ventricles are normal in size
    - No acute infarct on diffusion-weighted imaging
    """
)

response1 = agent_executor.invoke({
    "messages": [{"role": "user", "content": user_input}]
})
response_text = response1.get("output")
if response_text is None and "messages" in response1:
    response_text = response1["messages"][-1].content
if response_text is None:
    response_text = str(response1)

memory.save_context(
    {"input": user_input},
    {"output": response_text}
 )

print("\n" + "=" * 100)
print("📤 AGENT RESPONSE:")
print("=" * 100)
print(response_text)

input_processor.reset()

🧪 TEST CASE 1: Basic Symptom Analysis

📤 AGENT RESPONSE:
### 📋 Case Summary
- **Symptoms**: Persistent headache for 5 days (left temporal region), Blurred vision in left eye, Mild nausea in the morning, Difficulty concentrating, Patient age: 45, Male
- **MRI/Scan Findings**: Small area of white matter hyperintensity in the left temporal lobe, No mass lesion identified, No midline shift, Ventricles are normal in size, No acute infarct on diffusion-weighted imaging

### 🔍 Analysis
The patient's symptoms of persistent headache, blurred vision, nausea, and difficulty concentrating, combined with the MRI findings of white matter hyperintensity, may suggest a neurological issue that requires further evaluation. The absence of mass lesions and acute infarcts is reassuring, but the hyperintensity could indicate other conditions such as small vessel disease or demyelination. It is important to correlate these findings with clinical symptoms and consider further diagnostic workup or referral to 

In [104]:
# Test Case 2: With Hospital Search Request

print("=" * 100)
print("🧪 TEST CASE 2: Case Analysis + Hospital Search")
print("=" * 100)

user_input2 = input_processor.build_agent_input(
    text_symptoms="""
    - Sharp chest pain during physical activity
    - Shortness of breath
    - Occasional dizziness
    - Family history of heart disease
    - Patient age: 55, Female
    """,
    mri_description="""
    Cardiac MRI Summary:
    - Left ventricular ejection fraction: 48% (mildly reduced)
    - Mild left ventricular hypertrophy
    - No pericardial effusion
    - Mild mitral regurgitation noted
    """,
    additional_request="Please also find cardiology hospitals near Chicago, IL"
)

response2 = agent_executor.invoke({
    "messages": [{"role": "user", "content": user_input2}]
})
response_text = response2.get("output")
if response_text is None and "messages" in response2:
    response_text = response2["messages"][-1].content
if response_text is None:
    response_text = str(response2)

memory.save_context(
    {"input": user_input2},
    {"output": response_text}
 )

print("\n" + "=" * 100)
print("📤 AGENT RESPONSE:")
print("=" * 100)
print(response_text)

input_processor.reset()


🧪 TEST CASE 2: Case Analysis + Hospital Search

📤 AGENT RESPONSE:
### 📋 Case Summary
- **Symptoms**: 
  - Sharp chest pain during physical activity
  - Shortness of breath
  - Occasional dizziness
  - Family history of heart disease
  - Patient age: 55, Female
- **MRI/Scan Findings**: 
  - Left ventricular ejection fraction: 48% (mildly reduced)
  - Mild left ventricular hypertrophy
  - No pericardial effusion
  - Mild mitral regurgitation noted

### 🔍 Analysis
The patient presents with symptoms indicative of potential cardiac issues, including sharp chest pain, shortness of breath, and dizziness, particularly during physical activity. The MRI findings show a mildly reduced left ventricular ejection fraction and mild left ventricular hypertrophy, which may suggest underlying heart conditions. The family history of heart disease further elevates the concern for cardiovascular issues.

### 📌 Recommended Actions
It is recommended to consult a cardiologist for further evaluation and manage

In [105]:
# Test Case 3: Text Only (No MRI)

print("=" * 100)
print("🧪 TEST CASE 3: Text-Only Symptom Analysis")
print("=" * 100)

user_input3 = input_processor.build_agent_input(
    text_symptoms="""
    - Persistent lower back pain for 2 weeks
    - Pain radiates to left leg
    - Numbness in left foot
    - Difficulty walking long distances
    - Patient age: 38, Male, Office worker
    """
)

response3 = agent_executor.invoke({
    "messages": [{"role": "user", "content": user_input3}]
})
response_text = response3.get("output")
if response_text is None and "messages" in response3:
    response_text = response3["messages"][-1].content
if response_text is None:
    response_text = str(response3)

memory.save_context(
    {"input": user_input3},
    {"output": response_text}
 )

print("\n" + "=" * 100)
print("📤 AGENT RESPONSE:")
print("=" * 100)
print(response_text)

input_processor.reset()

🧪 TEST CASE 3: Text-Only Symptom Analysis

📤 AGENT RESPONSE:
### 📋 Case Summary
- **Symptoms**: Persistent lower back pain for 2 weeks, Pain radiates to left leg, Numbness in left foot, Difficulty walking long distances, Patient age: 38, Male, Office worker
- **MRI/Scan Findings**: Not provided

### 🔍 Analysis
The patient presents with persistent lower back pain that radiates to the left leg, accompanied by numbness in the left foot and difficulty walking long distances. These symptoms may suggest nerve involvement, possibly due to a herniated disc or other spinal issues. The patient's occupation as an office worker may contribute to prolonged sitting, which can exacerbate back pain.

### 📌 Recommended Actions
Consult a healthcare professional for a thorough evaluation, including a physical examination and possibly imaging studies such as an MRI to assess spinal health.

### ⚠️ Disclaimer
*This is not a medical diagnosis. Consult a doctor.*

---

The case data has been saved successful

In [106]:
# View All Stored Cases in CSV

print("📊 All Stored Patient Cases")
print("=" * 100)

df = pd.read_csv(CSV_FILE_PATH)
print(f"Total cases stored: {len(df)}\n")

for idx, row in df.iterrows():
    print(f"📋 {row['case_id']} | {row['timestamp']}")
    print(f"   Symptoms: {row['patient_symptoms'][:80]}...")
    print(f"   MRI: {row['mri_description'][:80]}...")
    print(f"   Summary: {row['ai_summary'][:80]}...")
    print(f"   Actions: {row['recommended_actions'][:80]}...")
    print(f"   Disclaimer: {row['disclaimer']}")
    print("-" * 100)

df

📊 All Stored Patient Cases
Total cases stored: 14

📋 CASE-20260424145722 | 2026-04-24T14:57:22.590459
   Symptoms: Severe headache, blurred vision...
   MRI: Mild white matter hyperintensities...
   Summary: Patient presents with neurological symptoms...
   Actions: Consult neurologist...
   Disclaimer: This is not a medical diagnosis. Consult a doctor.
----------------------------------------------------------------------------------------------------
📋 CASE-20260424145731 | 2026-04-24T14:57:31.308154
   Symptoms: Severe headache, blurred vision...
   MRI: Mild white matter hyperintensities...
   Summary: Patient presents with neurological symptoms...
   Actions: Consult neurologist...
   Disclaimer: This is not a medical diagnosis. Consult a doctor.
----------------------------------------------------------------------------------------------------
📋 CASE-20260424145959 | 2026-04-24T14:59:59.546426
   Symptoms: Severe headache, blurred vision...
   MRI: Mild white matter hyperintensi

,case_id,timestamp,patient_symptoms,mri_description,ai_summary,recommended_actions,disclaimer
0,CASE-20260424145722,2026-04-24T14:57:22.590459,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
1,CASE-20260424145731,2026-04-24T14:57:31.308154,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
2,CASE-20260424145959,2026-04-24T14:59:59.546426,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
3,CASE-20260424150637,2026-04-24T15:06:37.689787,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
4,CASE-20260424151525,2026-04-24T15:15:25.900821,"Severe headache, blurred vision",Mild white matter hyperintensities,Patient presents with neurological symptoms,Consult neurologist,This is not a medical diagnosis. Consult a doc...
5,CASE-20260424152157,2026-04-24T15:21:57.890933,"Sharp chest pain during physical activity, Sho...",Left ventricular ejection fraction: 48% (mildl...,The patient presents with symptoms indicative ...,It is advisable for the patient to consult a c...,This is not a medical diagnosis. Consult a doc...
6,CASE-20260424152214,2026-04-24T15:22:14.428974,Persistent headache for 5 days (left temporal ...,Small area of white matter hyperintensity in t...,The patient presents with a combination of neu...,Consult a neurologist for further evaluation a...,This is not a medical diagnosis. Consult a doc...
7,CASE-20260424152321,2026-04-24T15:23:21.400827,"Persistent lower back pain for 2 weeks, pain r...",No MRI/Scan findings provided,The symptoms described suggest possible issues...,"Consult a healthcare professional, preferably ...",This is not a medical diagnosis. Consult a doc...
8,CASE-20260424152331,2026-04-24T15:23:31.332173,"Persistent lower back pain for 2 weeks, Pain r...",Not provided,The symptoms suggest possible nerve involvemen...,"Consult a healthcare professional, possibly a ...",This is not a medical diagnosis. Consult a doc...
9,CASE-20260424152635,2026-04-24T15:26:35.311464,Persistent headache for 5 days (left temporal ...,Small area of white matter hyperintensity in t...,"The patient's symptoms of persistent headache,...",Consult a neurologist for further evaluation a...,This is not a medical diagnosis. Consult a doc...


In [107]:
# Inspect Conversation Memory

print("🧠 Current Memory State")
print("=" * 100)

memory_data = memory.load_memory_variables({})
history = memory_data.get("chat_history", [])

print("\n📝 Chat History (Summary + Recent Messages):")
print("-" * 100)

if not history:
    print("(empty)")
else:
    for idx, entry in enumerate(history, start=1):
        if isinstance(entry, dict):
            user_text = entry.get("inputs", {}).get("input", "")
            ai_text = entry.get("outputs", {}).get("output", "")
            print(f"\nTurn {idx} User: {user_text[:200]}...")
            print(f"Turn {idx} AI: {ai_text[:200]}...")
        else:
            role = entry.__class__.__name__.replace("Message", "")
            content = getattr(entry, "content", str(entry))
            print(f"\n[{role}]: {content[:200]}...")

if hasattr(memory, 'moving_summary_buffer'):
    print(f"\n\n📄 Moving Summary Buffer:")
    print("-" * 100)
    print(memory.moving_summary_buffer or "Empty")

🧠 Current Memory State

📝 Chat History (Summary + Recent Messages):
----------------------------------------------------------------------------------------------------

Turn 1 User: ## Patient Symptoms:

    - Persistent headache for 5 days (left temporal region)
    - Blurred vision in left eye
    - Mild nausea in the morning
    - Difficulty concentrating
    - Patient age: 45...
Turn 1 AI: ### 📋 Case Summary
- **Symptoms**: Persistent headache for 5 days (left temporal region), Blurred vision in left eye, Mild nausea in the morning, Difficulty concentrating, Patient age: 45, Male
- **MR...

Turn 2 User: ## Patient Symptoms:

    - Sharp chest pain during physical activity
    - Shortness of breath
    - Occasional dizziness
    - Family history of heart disease
    - Patient age: 55, Female
    

## ...
Turn 2 AI: ### 📋 Case Summary
- **Symptoms**: 
  - Sharp chest pain during physical activity
  - Shortness of breath
  - Occasional dizziness
  - Family history of heart disease
  

In [108]:
# Message Trimming Demonstration

print("✂️ Message Trimming Demonstration")
print("=" * 100)

# Simulate a long conversation
long_conversation = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="I have severe headaches."),
    AIMessage(content="I understand you're experiencing severe headaches. "
                      "Can you tell me more about the duration and location?"),
    HumanMessage(content="It's been 5 days, left side of head."),
    AIMessage(content="Thank you. Any other symptoms like nausea or vision changes?"),
    HumanMessage(content="Yes, blurred vision and nausea."),
    AIMessage(content="I see. Do you have any MRI or scan results?"),
    HumanMessage(content="Yes, the MRI shows white matter hyperintensities."),
    AIMessage(content="Based on the symptoms and MRI findings..."),
    HumanMessage(content="Can you find hospitals near me in Boston?"),
    AIMessage(content="Here are some hospitals near Boston..."),
    HumanMessage(content="Please save this case."),
    AIMessage(content="Case has been saved successfully."),
]

print(f"Original message count: {len(long_conversation)}")

# Trim to fit context window
trimmed_messages = trim_conversation_messages(
    long_conversation,
    max_tokens=500
)

print(f"Trimmed message count: {len(trimmed_messages)}")
print(f"\n📨 Retained Messages:")
for msg in trimmed_messages:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}]: {msg.content[:100]}...")

✂️ Message Trimming Demonstration
Original message count: 13
Trimmed message count: 13

📨 Retained Messages:
  [System]: 
You are a Multi-Modal Medical AI Assistant Agent. Your role is to help
analyze patient cases based ...
  [Human]: I have severe headaches....
  [AI]: I understand you're experiencing severe headaches. Can you tell me more about the duration and locat...
  [Human]: It's been 5 days, left side of head....
  [AI]: Thank you. Any other symptoms like nausea or vision changes?...
  [Human]: Yes, blurred vision and nausea....
  [AI]: I see. Do you have any MRI or scan results?...
  [Human]: Yes, the MRI shows white matter hyperintensities....
  [AI]: Based on the symptoms and MRI findings......
  [Human]: Can you find hospitals near me in Boston?...
  [AI]: Here are some hospitals near Boston......
  [Human]: Please save this case....
  [AI]: Case has been saved successfully....


In [109]:
# Interactive Chat Interface

def run_interactive_chat():
    """
    Run an interactive chat session with the Medical AI Agent.
    Type 'quit' to exit, 'reset' to clear memory, 'cases' to view stored cases.
    """
    print("=" * 100)
    print("🏥 Medical AI Assistant - Interactive Chat")
    print("=" * 100)
    print("Commands:")
    print("  'quit'   - Exit the chat")
    print("  'reset'  - Clear conversation memory")
    print("  'cases'  - View stored patient cases")
    print("  'memory' - View current memory state")
    print("=" * 100)
    
    while True:
        print()
        user_input = input("👤 You: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() == 'quit':
            print("👋 Goodbye! Remember to consult a real doctor.")
            break
        
        if user_input.lower() == 'reset':
            memory.clear()
            print("🔄 Memory cleared!")
            continue
        
        if user_input.lower() == 'cases':
            if os.path.exists(CSV_FILE_PATH):
                df = pd.read_csv(CSV_FILE_PATH)
                print(f"\n📊 Stored Cases ({len(df)} total):")
                print(df[['case_id', 'timestamp', 'patient_symptoms']].to_string())
            else:
                print("No cases stored yet.")
            continue
        
        if user_input.lower() == 'memory':
            mem_data = memory.load_memory_variables({})
            history = mem_data.get("chat_history", [])
            if not history:
                print("(empty)")
            else:
                for idx, entry in enumerate(history, start=1):
                    if isinstance(entry, dict):
                        user_text = entry.get("inputs", {}).get("input", "")
                        ai_text = entry.get("outputs", {}).get("output", "")
                        print(f"\nTurn {idx} User: {user_text[:120]}...")
                        print(f"Turn {idx} AI: {ai_text[:120]}...")
                    else:
                        role = entry.__class__.__name__.replace("Message", "")
                        content = getattr(entry, "content", str(entry))
                        print(f"  [{role}]: {content[:120]}...")
            continue
        
        # Run agent
        try:
            response = agent_executor.invoke({
                "messages": [{"role": "user", "content": user_input}]
            })
            response_text = response.get("output")
            if response_text is None and "messages" in response:
                response_text = response["messages"][-1].content
            if response_text is None:
                response_text = str(response)

            memory.save_context(
                {"input": user_input},
                {"output": response_text}
            )

            print(f"\n🤖 Assistant: {response_text}")
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")

In [110]:
# Start the interactive chat
run_interactive_chat()

🏥 Medical AI Assistant - Interactive Chat
Commands:
  'quit'   - Exit the chat
  'reset'  - Clear conversation memory
  'cases'  - View stored patient cases
  'memory' - View current memory state


📊 Stored Cases (14 total):
                case_id                   timestamp                                                                                                                                                   patient_symptoms
0   CASE-20260424145722  2026-04-24T14:57:22.590459                                                                                                                                    Severe headache, blurred vision
1   CASE-20260424145731  2026-04-24T14:57:31.308154                                                                                                                                    Severe headache, blurred vision
2   CASE-20260424145959  2026-04-24T14:59:59.546426                                                                               